In [5]:
import pandas as pd
from pathlib import Path
import os
# Get current working directory and navigate to src
current_dir = Path(os.getcwd())
# Assuming you're running from /data/home/fshachar/Readability/src/Analysis/
src_path = current_dir.parent  # Go up to src/
df = pd.read_csv(src_path / 'data/OneStop/OneStop_mean_ci_stats.csv')

data used:

In [ ]:
# text comes from:
# ----------------------------------------------
# src_path = Path.cwd().parents[0]
# parag_metrics_path = src_path / Path("Readability/src/readability_metrics/data/paragraphs_metrics_cleaned.csv")

# data_parags = pd.read_csv(parag_metrics_path)
# ------------------------------------------------
# word properties (word len, frequency, surprisal) come from:
# ----------------------------------------------
# from src.constants import EYE_BY_WORD_ALIGNED_PATHS
# EYE_BY_WORD_DF_L1_ALIGNED_PATH = EYE_BY_WORD_ALIGNED_PATHS['L1']
# eye_data = pd.read_csv(EYE_BY_WORD_DF_L1_ALIGNED_PATH)
# eye_data_one_version = eye_data.drop_duplicates(subset=["unique_paragraph_id", "level", "text_spacing_version","gpt2_surprisal","pythia70m_surprisal","word_length", "word_length_no_punctuation", "wordfreq_frequency", "text_spacing_version", "IA_LABEL"])
# eye_data_one_version = eye_data_one_version[eye_data_one_version["text_spacing_version"]==0]

# agg_eye = eye_data_one_version.groupby(['text_id', 'level']).agg({
#     'wordfreq_frequency': 'mean',
#     'word_length': 'mean',
#     'gpt2_surprisal': 'mean',
#     'pythia70m_surprisal': 'mean'
# }).reset_index()



Paragraph general stats

In [17]:
#calculation:
# calculate number of paragraphs per level
# num_parags_adv = len(data_parags[data_parags['level'] == 'Adv'])
# num_parags_ele = len(data_parags[data_parags['level'] == 'Ele'])
# # calculate number of sentences per level
# num_sentences_adv = data_parags[data_parags['level'] == 'Adv']['n_sentences'].sum()
# num_sentences_ele = data_parags[data_parags['level'] == 'Ele']['n_sentences'].sum()

# number of question is the same for both levels - and known so hardcoded 
num_questions = 486

# printing from csv, in table format:
number_of_passages_orig = df[(df.metric == "number_of_paragraphs") & (df.group == "original")]["mean"].values[0]
number_of_passages_simp = df[(df.metric == "number_of_paragraphs") & (df.group == "simplified")]["mean"].values[0]
number_of_sentences_orig = df[(df.metric == "number_of_sentences") & (df.group == "original")]["mean"].values[0]
number_of_sentences_simp = df[(df.metric == "number_of_sentences") & (df.group == "simplified")]["mean"].values[0]

import pandas as pd

# Create a summary table
summary_data = {
    'Statistic': [
        'Number of passages',
        'Number of sentences', 
        'Number of questions'
    ],
    'Original': [
        int(number_of_passages_orig),
        int(number_of_sentences_orig),
        num_questions
    ],
    'Simplified': [
        int(number_of_passages_simp),
        int(number_of_sentences_simp), 
        num_questions
    ]
}

summary_table = pd.DataFrame(summary_data)

# Or for a nicer display in Jupyter:
display(summary_table)

,Statistic,Original,Simplified
0,Number of passages,162,162
1,Number of sentences,936,931
2,Number of questions,486,486


per passage statistics

In [24]:
# data_parags['n_sentences'] = data_parags['text'].apply(count_sentences)

# # ----------------------- per paragraph stats (paired) ---------------------------
# # run paired t-test on number of sentences per paragraph
# number_of_sentences_per_parag_dict = paired_ttest_from_data(data_parags, text_id_col="text_id", level_col="level", value="n_sentences", label="sentences per paragraph")
# # run paired t-test on number of words per paragraph
# number_of_words_per_parag_dict = paired_ttest_from_data(data_parags, text_id_col="text_id", level_col="level", value="n_words", label="words per paragraph")


# Filter for the metrics you want
metrics_to_show = ["sentences_per_passage", "words_per_passage"]
filtered_df = df[df.metric.isin(metrics_to_show)]

# Create a pivot table with Original vs Simplified as columns
per_passage_table = filtered_df.pivot_table(
    index='metric', 
    columns='group', 
    values=['mean_ci', 'pval', 'stars'], 
    aggfunc='first'
)

# Flatten column names and reorder
per_passage_table.columns = [f'{col[1]}_{col[0]}' for col in per_passage_table.columns]
per_passage_table = per_passage_table[['original_mean_ci', 'simplified_mean_ci','original_pval', 'original_stars']]

# Rename columns for clarity
per_passage_table.columns = ['Original', 'Simplified', 'pval', 'significance']

display(per_passage_table)

,Original,Simplified,pval,significance
metric,,,,
sentences_per_passage,5.78 ± 0.31,5.75 ± 0.27,7.704514e-01,ns
words_per_passage,119.92 ± 4.33,97.14 ± 3.66,6.638134e-35,***


In [25]:

# ----------------------- per sentence stats (unpaired) ---------------------------
# sentence_lengths_adv = get_sentence_lens_for_level(data_parags, level="Adv")
# sentence_lengths_ele = get_sentence_lens_for_level(data_parags, level="Ele")

# sentence_length_dict = unpaired_ttest(sentence_lengths_adv, sentence_lengths_ele, label="sentence length")
# Build and print sentence length table from csv
sentence_length_df = df[df["metric"] == "sentence_length_words"]

sentence_length_table = sentence_length_df.pivot_table(
    index="metric",
    columns="group",
    values=["mean_ci", "pval", "stars"],
    aggfunc="first"
)

sentence_length_table.columns = [f"{col[1]}_{col[0]}" for col in sentence_length_table.columns]
sentence_length_table = sentence_length_table[
    ["original_mean_ci", "simplified_mean_ci", "original_pval", "original_stars"]
]
sentence_length_table.columns = ["Original", "Simplified", "pval", "significance"]
sentence_length_table.index = ["sentence_length_words"]

print(sentence_length_table)


                           Original    Simplified          pval significance
sentence_length_words  20.76 ± 0.66  16.90 ± 0.51  2.748215e-19          ***


In [26]:
# # calc unpaired t test on word length, word frequency and surprisal using eye_data_one_version:

# word_length_adv = eye_data_one_version[eye_data_one_version['level'] == 'Adv']['word_length']
# word_length_ele = eye_data_one_version[eye_data_one_version['level'] == 'Ele']['word_length']
# word_length_dict = unpaired_ttest(word_length_adv, word_length_ele, label="word length")

# wordfreq_adv = eye_data_one_version[eye_data_one_version['level'] == 'Adv']['wordfreq_frequency']
# wordfreq_ele = eye_data_one_version[eye_data_one_version['level'] == 'Ele']['wordfreq_frequency']
# word_freq_dict = unpaired_ttest(wordfreq_adv, wordfreq_ele, label="word frequency")

# pythia_surprisal_adv = eye_data_one_version[eye_data_one_version['level'] == 'Adv']['pythia70m_surprisal']
# pythia_surprisal_ele = eye_data_one_version[eye_data_one_version['level'] == 'Ele']['pythia70m_surprisal']
# pythia70m_surprisal_dict = unpaired_ttest(pythia_surprisal_adv, pythia_surprisal_ele, label="surprisal")

# Build word-level statistics table (from csv) for:
# word length, word frequency, and surprisal
word_metrics = [
    "word_length_characters",
    "word_frequency_wordfreq",
    "word_surprisal_pythia70m",
]

word_stats_df = df[df["metric"].isin(word_metrics)]

word_stats_table = word_stats_df.pivot_table(
    index="metric",
    columns="group",
    values=["mean_ci", "pval", "stars"],
    aggfunc="first",
)

word_stats_table.columns = [f"{col[1]}_{col[0]}" for col in word_stats_table.columns]
word_stats_table = word_stats_table[
    ["original_mean_ci", "simplified_mean_ci", "original_pval", "original_stars"]
]
word_stats_table.columns = ["Original", "Simplified", "pval", "significance"]

word_stats_table = word_stats_table.rename(
    index={
        "word_length_characters": "word_length_characters",
        "word_frequency_wordfreq": "word_frequency_wordfreq",
        "word_surprisal_pythia70m": "word_surprisal_pythia70m",
    }
)

display(word_stats_table)


,Original,Simplified,pval,significance
metric,,,,
word_frequency_wordfreq,11.26 ± 0.08,10.98 ± 0.08,2.084327e-06,***
word_length_characters,4.91 ± 0.04,4.72 ± 0.04,1.972161e-11,***
word_surprisal_pythia70m,5.01 ± 0.06,4.77 ± 0.06,1.305925e-08,***
